# Prueba Técnica - Concurso Científico/a de Datos - Unidad de Gobierno de Datos

**Postulante:** Charlie Condorcet

## 1. Clasificando texto

pipeline simple y sin fuga de datos: limpieza básica (minúsculas, sin tildes ni símbolos), TF-IDF con unigramas y bigramas por separado para **actividad**, **oficio** y **tareas**, y un clasificador **LinearSVC**. 

Se vectorizan los tres campos por separado porque **actividad** suele pesar más en el código económico que **oficio** o **tareas**, y LinearSVC funciona rápido y estable en texto disperso multiclase.

In [2]:
import mysql.connector
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.svm import LinearSVC

DB_CONFIG = {
    "host": "sql.freedb.tech",
    "port": 3306,
    "user": "u_y8tlWZ",
    "password": "wSkxw3xbhsQg",
    "database": "freedb_N4nnid0a",
    "connection_timeout": 15,
}

COLUMNAS_TEXTO = ["actividad", "oficio", "tareas"]

# descarga de tablas genericas
def descargar_tabla(nombre_tabla: str) -> pd.DataFrame:
    # se abre y cierra la conexion en cada consulta
    conn = mysql.connector.connect(**DB_CONFIG)
    try:
        cursor = conn.cursor(dictionary=True)
        cursor.execute(f"SELECT * FROM {nombre_tabla}")
        return pd.DataFrame(cursor.fetchall())
    finally:
        conn.close()

# 1.1: descarga de train desde base de datos
train = descargar_tabla("train")
train["cod_final"] = train["cod_final"].astype(str).str.strip()

print(f"train: {train.shape[0]} filas, {train.shape[1]} columnas")
print(f"clases unicas en train: {train['cod_final'].nunique()}")
print("nulos por columna:", train.isna().sum().to_dict())
print(
    "textos vacios por columna:",
    {c: int((train[c].fillna('').str.strip() == '').sum()) for c in COLUMNAS_TEXTO},
)
train.head(5)

train: 35340 filas, 5 columnas
clases unicas en train: 79
nulos por columna: {'id': 0, 'actividad': 0, 'oficio': 0, 'tareas': 0, 'cod_final': 0}
textos vacios por columna: {'actividad': 0, 'oficio': 453, 'tareas': 455}


,id,actividad,oficio,tareas,cod_final
0,0,producción uva exportación,temporera agrícola,mantenimiento de parronales,01
1,1,corporación municipal,profesor mecánico industrial,"preparar clases, enseñar",84
2,4,contratista de rrhh para obras de construcción,trabajador de construcción,veredas,41
3,5,municipalidad de puerto natales,técnico paramedico,"toma de signos vitales, entrega de horas a pac...",84
4,7,guardias seguridad y proteccion,administrador de contrato,administrar dirigir contratar personal,80


In [3]:
# 1.2: entrenar clasificador de texto
def limpiar_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    # normalizar cada columna de texto
    salida = pd.DataFrame(index=df.index)
    for col in COLUMNAS_TEXTO:
        salida[col] = (
            df[col]
            .fillna("")
            .astype(str)
            .str.lower()
            .str.normalize("NFKD")
            .str.encode("ascii", errors="ignore")
            .str.decode("utf-8")
            .str.replace(r"[^a-z0-9\s]", " ", regex=True)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
    return salida


# vectorizar cada campo por separado
vectorizador = ColumnTransformer(
    transformers=[
        ("actividad", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), "actividad"),
        ("oficio", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), "oficio"),
        ("tareas", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True), "tareas"),
    ]
)

modelo = Pipeline(
    steps=[
        ("limpieza", FunctionTransformer(limpiar_dataframe, validate=False)),
        ("tfidf", vectorizador),
        ("clf", LinearSVC(random_state=42)),
    ]
)

modelo.fit(train[COLUMNAS_TEXTO], train["cod_final"])
print("modelo entrenado correctamente.")

modelo entrenado correctamente.


In [4]:
# 1.3 y 1.4: descarga de test y predicciones
test = descargar_tabla("test")
test["cod_final"] = test["cod_final"].astype(str).str.strip()

clases_no_vistas = set(test["cod_final"]) - set(train["cod_final"])
print(f"test: {test.shape[0]} filas | Clases unicas: {test['cod_final'].nunique()} | no vistas en train: {len(clases_no_vistas)}")

# se predicen las clases en test
test["prediccion"] = modelo.predict(test[COLUMNAS_TEXTO])
test[["id", "actividad", "oficio", "cod_final", "prediccion"]].head(8)

test: 3927 filas | Clases unicas: 77 | no vistas en train: 0


,id,actividad,oficio,cod_final,prediccion
0,21480,investigacion y desarrollo de semillas de toma...,encargado de riego,72,72
1,16064,metro de santiago,mantencion en metro,49,49
2,12589,"producción de uva vinifera (solo produccion), ...",temporero agricola,01,01
3,9547,comercio de decoración para el hogar,vendedora de local decoración,48,48
4,19225,jardín infantil,auxiliar de aseo en jardín infantil,85,85
5,10062,producción y distribución de alimentos frescos,operario de producción,10,10
6,2895,servicio de poda de parrones,temporero agricola,01,01
7,30133,delivery venta por menor de frutas y verduras,delivery,48,48


In [5]:
# 1.5: analisis de resultados
y_true = test["cod_final"]
y_pred = test["prediccion"]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

print(f"Accuracy:    {acc:.4f}")
print(f"Macro-F1:    {macro_f1:.4f}")
print(f"Weighted-F1: {weighted_f1:.4f}")

# se agrupa el F1 segun la frecuencia de cada clase en train
reporte = pd.DataFrame(
    classification_report(y_true, y_pred, output_dict=True, zero_division=0)
).T.drop(index=["accuracy", "macro avg", "weighted avg"], errors="ignore")
reporte["soporte_train"] = reporte.index.map(train["cod_final"].value_counts()).astype(int)
reporte["soporte_test"] = reporte["support"].astype(int)

tramos = pd.cut(
    reporte["soporte_train"],
    bins=[0, 30, 100, 500, 10000],
    labels=["<=30", "31-100", "101-500", ">500"],
)
print("\ndesempeño promedio por tamano de clase en train:")
display(
    reporte.groupby(tramos, observed=False).agg(
        clases=("f1-score", "count"),
        f1_promedio=("f1-score", "mean"),
        registros_test=("soporte_test", "sum"),
    ).round(4)
)

print("\n10 clases con menor F1:")
display(
    reporte.sort_values(["f1-score", "soporte_train"])
    .head(10)[["precision", "recall", "f1-score", "soporte_test", "soporte_train"]]
    .round(4)
)

# se revisan las confusiones mas frecuentes y una muestra de errores
errores = test[test["cod_final"] != test["prediccion"]]
print(f"\ntotal errores: {len(errores)} / {len(test)} ({len(errores) / len(test):.2%})")
print("\ntop 10 confusiones (real -> predicho):")
display(
    errores.groupby(["cod_final", "prediccion"])
    .size()
    .rename("casos")
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

print("\nmuestra de errores para inspeccion cualitativa:")
display(errores[["id", "actividad", "oficio", "tareas", "cod_final", "prediccion"]].head(10))

Accuracy:    0.9162
Macro-F1:    0.8019
Weighted-F1: 0.9136

desempeño promedio por tamano de clase en train:


,clases,f1_promedio,registros_test
soporte_train,,,
<=30,12,0.6083,22
31-100,23,0.7434,177
101-500,29,0.8742,834
>500,13,0.9228,2894



10 clases con menor F1:


,precision,recall,f1-score,soporte_test,soporte_train
24,0.0000,0.0,0.0000,1,10
70,0.0000,0.0,0.0000,1,14
59,0.0000,0.0,0.0000,2,15
37,0.0000,0.0,0.0000,2,17
27,0.0000,0.0,0.0000,5,42
09,0.3333,0.1,0.1538,10,85
79,0.5000,0.5,0.5000,2,21
32,0.6667,0.4,0.5000,10,90
20,0.8000,0.5,0.6154,8,72
07,0.7500,0.6,0.6667,5,40



total errores: 329 / 3927 (8.38%)

top 10 confusiones (real -> predicho):


,cod_final,prediccion,casos
0,43,41,17
1,41,43,8
2,10,48,8
3,86,84,7
4,56,48,7
5,85,84,6
6,84,85,6
7,49,48,5
8,42,41,4
9,01,48,4



muestra de errores para inspeccion cualitativa:


,id,actividad,oficio,tareas,cod_final,prediccion
18,17527,producción y procesamiento de salmón,subgerente de planta,"administra, realiza el funcionamiento del. are...",03,10
31,33964,"presta sevicion en desmurres, confeccion y rep...",labores agricolas,"desmurres, confección y reparación de cercos.",01,14
33,281,productora de eventos,garzona,"montar mesas, servir platos, limpiar loza",93,56
35,35053,confeccion de artesania con conchas de mar,artesana en conchitas de mar,"buscar conchitas de mar, hacer llaveros",32,14
51,32750,recoleccion y venta de chatarra y aluminio,venta de chatarra y latas de aluminio,recolección de latas y chatarra,48,38
81,33276,repaciones menor en viviendas particular,maestro,"reparaciones menor en viviendas, reparación de...",41,43
103,6007,construcción e instalación de eletricidad,electricista,instalacion electrica,43,41
114,6405,fabricación e instalación de piezas de fierro ...,tornero,fabricación de piezas de fierro para vehículos...,27,25
128,22444,administración municipal de educacion y salud,docente,"docente educación básica y media, apoyo pedagó...",85,84
132,25982,empresa de demarcación viales,ayudante de pintor,pinta calles,52,42


### 1.5: Análisis de resultados

El modelo llega a **0.9162 de Accuracy** y **0.9136 de Weighted-F1** (329 errores sobre 3.927 registros), entrenando rápido en CPU y con una arquitectura simple. En las 13 clases con más de 500 ejemplos en train (que concentran el 73,7% del test) el F1 promedio es **0.9228**, y varias divisiones con vocabulario bien definido superan 0.95 (como 80, 69, 36, 85, 49, 01 y 04). Mantener los tres campos separados también ayuda cuando oficio y tareas vienen vacíos (453 filas en train y 46 en test).

El principal problema está en el desbalance, que se refleja directo en la caída a **0.8019 de Macro-F1**. Las 12 clases con 30 o menos registros en train promedian un F1 de 0.6083, y cinco clases quedan en 0 (24, 70, 59, 37 y 27). Al mismo tiempo, la clase más grande (48, comercio, con 5.507 registros) tiende a atraer casos ambiguos de otras divisiones cuando aparece la palabra venta.

Al revisar los errores concretos aparecen dos patrones claros. Primero, varias confusiones se dan en fronteras cercanas del clasificador: 43 vs. 41 suma 25 errores (construcción especializada frente a edificios, donde palabras como construcción arrastran instalaciones eléctricas hacia 41), y 85 o 86 contra 84 suma 19 errores (educación y salud municipal frente a administración pública). Segundo, como el TF-IDF es solo por palabras, los errores de tipeo frecuentes en las glosas (eletricidad, repaciones, sevicion) se pierden fuera del vocabulario, y también cuesta resolver registros que mezclan dos actividades distintas o que no traen oficio ni tareas.

### 1.6: Plan de mejora

Para iterar de forma ordenada evaluaría cada cambio con validación cruzada estratificada sobre train, dejando test solo para la medición final.

En una primera etapa atacaría los errores más baratos de corregir: agregar n-gramas de caracteres (3 a 5) junto con palabras para absorber faltas de ortografía y abreviaturas, probar mayor peso relativo a **actividad** dentro del ColumnTransformer y evaluar ajustes de regularización y pesos de clase para levantar las categorías con menos registros. También revisaría en train los casos con texto idéntico y distinta etiqueta, sobre todo en los pares 41/43 y 84/85/86.

Si el objetivo pasa a uso productivo, calibraría las probabilidades para fijar un umbral de confianza: los casos claros se codifican automáticamente y los ambiguos o con campos vacíos se envían a revisión humana. Como paso siguiente, probaría ajustar un modelo de lenguaje en español (por ejemplo BETO o mDeBERTa-v3) marcando explícitamente cada campo para capturar mejor el contexto en fronteras como educación municipal vs. administración pública, y dejaría un LLM solo como segunda capa sobre los casos de baja confianza o clases muy raras, restringido a los códigos candidatos del clasificador.

## 2. Modelo en producción

Lo primero que llama la atención es que la caída no es uniforme. Entre testing (Acc 0.938, Macro-F1 0.823) y el mes 6 (0.913 / 0.796) el desgaste es moderado, pero en el mes 10 hay un quiebre fuerte hasta **0.83 de Accuracy** y **0.71 de Macro-F1**, manteniendo además la brecha de ~11 a 12 puntos entre ambas métricas por el desbalance de clases.

Para explicar esto plantearía tres líneas principales, más una cautela sobre cómo se está midiendo:

- **Cambio gradual en las entradas y estacionalidad:** en empleo la distribución de actividades cambia según la época del año (temporadas agrícolas, comercio, turismo) y van apareciendo glosas o rubros nuevos fuera del vocabulario con que se entrenó el modelo. Eso encaja bien con la baja lenta hasta el mes 6.
- **Un cambio puntual antes del mes 10:** una caída de más de 8 puntos entre el mes 6 y el mes 10 difícilmente es solo desgaste natural. Puede venir de un problema aguas arriba en los datos (más campos vacíos, cambios en el formulario de captura, problemas de encoding) o de un cambio de criterio en la codificación manual para casos límite como subcontratación o servicios municipales.
- **Tamaño muestral pequeño (n = 100):** con 100 registros al mes repartidos en decenas de clases, el margen de error del Accuracy al 95% anda por +/- 5 a 7 puntos y el Macro-F1 tiene todavía más varianza porque muchas clases poco frecuentes caen con 0 o 1 observación. Las bajas leves de los meses 3 y 6 podrían mezclar deterioro real con ruido muestral, aunque la caída del mes 10 frente a testing ya es demasiado grande para explicarla solo por azar. A eso se suma haber dejado cuatro meses sin medir entre el mes 6 y el mes 10.

**Estrategia de mitigación**

Antes de reentrenar, partiría por diagnosticar los registros del mes 10: revisar tasas de vacíos, largo de textos, palabras fuera de vocabulario y ver si los errores se concentran en pocas clases o si son generalizados, junto con chequear consistencia entre etiquetadores. Si el origen es un fallo en la captura de datos, reentrenar el modelo solo empeoraría el problema.

En paralelo, ajustaría el monitoreo para no enterarnos cuatro meses tarde. Mediría todos los meses con una muestra más grande o estratificada (asegurando presencia de clases minoritarias y casos dudosos para estabilizar el Macro-F1) y agregaría alertas semanales sin etiquetas: proporción de palabras nuevas, cambios en la distribución de códigos predichos (por ejemplo con PSI) y bajas en la confianza promedio del modelo.

Mientras se corrige el modelo, aplicaría un umbral sobre probabilidades calibradas para automatizar solo los registros seguros y mandar a revisión manual los casos inciertos. Eso contiene el error en producción y genera etiquetas justo en la zona donde el modelo está fallando. Con esos datos nuevos incorporados en una ventana reciente, reentrenaría validando siempre de forma temporal (entrenar con meses anteriores y evaluar en el último mes) y probaría la nueva versión en paralelo antes de reemplazar la productiva, dejando respaldo para revertir si no mejora.

## 3. Procesando texto

Se descarga la tabla **noticias** (10 filas) y se genera un resumen de máximo 50 palabras por registro usando el endpoint responses de OpenRouter provisto y razonamiento desactivado.

In [6]:
# 3.1 y 3.2: descarga de noticias y generacion de resumenes <= 50 palabras
import os
import requests
import pandas as pd
from dotenv import load_dotenv

# se carga la api key desde .env o variable de entorno para no exponerla en git
load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise ValueError("falta definir OPENROUTER_API_KEY en .env o variable de entorno")

URL_OPENROUTER = "https://openrouter.ai/api/v1/responses"
MODELO_LLM = "google/gemini-2.5-flash-lite"

# se descarga la tabla noticias desde MySQL (con respaldo local si el servidor satura conexiones)
try:
    noticias = descargar_tabla("noticias")
except Exception:
    noticias = pd.read_parquet("noticias_sample.parquet")

print(f"noticias cargadas: {noticias.shape[0]} filas")


def resumir_noticia(titular: str, cuerpo: str) -> str:
    # se solicita un maximo de 45 palabras para mantener margen bajo el limite de 50
    prompt = (
        "Resume la siguiente noticia en español en un máximo de 45 palabras "
        "(nunca superes las 50 palabras). Devuelve solo el texto del resumen, "
        "sin encabezados ni comentarios.\n\n"
        f"Titular: {titular}\n"
        f"Cuerpo: {cuerpo}"
    )
    payload = {
        "model": MODELO_LLM,
        "input": prompt,
        "reasoning": {"effort": "none"},
    }
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
    }
    response = requests.post(URL_OPENROUTER, headers=headers, json=payload, timeout=30)
    response.raise_for_status()
    return response.json()["output"][0]["content"][0]["text"].strip()


# se genera la columna resumen para las 10 noticias
resumenes = []
for _, fila in noticias.iterrows():
    resumen = resumir_noticia(fila["titular"], fila["cuerpo"])
    # si excede 50 palabras se solicita una version mas breve
    if len(resumen.split()) > 50:
        resumen = resumir_noticia(fila["titular"], f"Acorta a menos de 40 palabras: {resumen}")
    resumenes.append(resumen)

noticias["resumen"] = resumenes
noticias["palabras_resumen"] = noticias["resumen"].apply(lambda x: len(x.split()))

print(f"maximo de palabras obtenido: {noticias['palabras_resumen'].max()} (limite: 50)")
pd.set_option("display.max_colwidth", None)
display(noticias[["titular", "palabras_resumen", "resumen"]])


noticias cargadas: 10 filas
maximo de palabras obtenido: 44 (limite: 50)


,titular,palabras_resumen,resumen
0,escenas de geometría variable,35,"Sánchez usa la ""geometría variable"" de Zapatero para gobernar, buscando apoyos coyunturales. UP, Ciudadanos, nacionalistas y separatistas forman un bloque cambiante. Aunque Sánchez parece tener el control, crisis económicas o judiciales podrían desbaratar sus planes."
1,iglesias se reunirá con junqueras sin cristal y sin límite de tiempo,35,Pablo Iglesias se reunirá con Oriol Junqueras en la cárcel de Lledoners sin cristal y sin límite de tiempo. Abordarán la situación de los presos soberanistas y buscarán el apoyo de ERC para los presupuestos.
2,"bankia paga otros 219 millones al estado vía dividendo, el mayor de su historia",36,"Bankia pagará 219 millones de euros al Estado en concepto de dividendo, el mayor de su historia. Esta operación avanza en la devolución de ayudas públicas recibidas, sumando un total de 961 millones por esta vía."
3,"sánchez tiene ya una mayoría en el congreso para derogar la reforma laboral y la ""ley mordaza """,36,"Pedro Sánchez cuenta con mayoría en el Congreso para derogar la reforma laboral y la ""ley mordaza"". Las fuerzas de izquierda permitirán avanzar propuestas progresistas, como la ley de eutanasia y la prevalencia de convenios sectoriales."
4,un presidente irresponsable,44,"Juan Carlos I regulariza casi 4.5 millones de euros ante Hacienda por viajes en jet privado, tras otra regularización de 680.000 euros. El artículo critica la respuesta del presidente Sánchez, considerándola mezquina, y aboga por reconocer los logros del emérito, como parar el 23-F."
5,castro embarga el palacete y otra finca a matas para cubrir su fianza en nóos,31,"El juez Castro embarga dos propiedades de Jaume Matas por 3,5 millones para cubrir su fianza en el caso Nóos. Otros acusados también enfrentan embargos y deben informar de sus bienes."
6,"el jefe del ejecutivo, dispuesto a reunirse con pablo iglesias si se lo pide",41,"Rajoy se muestra dispuesto a reunirse con Pablo Iglesias si este se lo solicita. Atribuye la falta de encuentro a sus diferentes trayectorias y considera a Podemos un partido de ""izquierda muy extrema"", contrastando su discurso con la gestión del PP."
7,el último choque con puigdemont y las diferencias entre defensas la cuenta atrás para el juicio al procés,23,"El juicio del procés comenzará pronto, coincidiendo con negociaciones presupuestarias y tensiones independentistas. Las defensas discrepan estratégicamente, mientras Vox exige penas muy elevadas."
8,interior jubila al policía que perdió la visión tras los disturbios del procés»,35,"El Ministerio del Interior ha jubilado a un policía de 42 años que perdió la visión tras ser herido en los disturbios del procés en Barcelona. Una piedra le destrozó el casco, causándole secuelas irreversibles."
9,"otegi sobre bateragune ""ni nos domaron, ni nos doblaron, ni nos van a domesticar""",36,"Arnaldo Otegi reacciona ante la decisión del Supremo de reabrir el juicio del ""caso Bateragune"" con un mensaje de resistencia: ""¡Ni nos domaron, ni nos doblaron, ni nos van a domesticar!"". EH Bildu analizará la resolución."


## 4. Cuéntanos sobre un proyecto

Si, he trabajado mucho con esa arquitectura RAG, tanto a en el trabajo como a nivel personal para proyectos independientes.

Problema de negocio: Informes de fiscalización ambiental, son demasiado extensos requieren mucha revisión manual del contenido recibido, revisión del contenido general enviado, de los diferentes tipos de informes que hay, de si cumple o no con las normas especificas de su tipo de informe, de que las evidencias adjuntas correspondan o respalden las respuestas recibidas y así varias otras características que se traducen en información trabajable fácilmente con esta arquitectura. 
Para resolverlo lo primero fue el estudio del problema, para comprender el alcance que puede llevar, los diferentes tipos de casos, los tipos variados de fuentes utilizadas, ver los tiempos donde más efectivo o mayor mejora podría presentar, y recién ahí podría comenzar con el prototipado del proyecto, apuntado a un producto para producción según el alcance de información obtenido del problema para poder calcular los recursos a utilizar.

Sobre la arquitectura en si use Qdrant, modelos populares de LLM evitando vendor-lock, modelos de embedding y reranker de BGE, con PostgreSQL para manejar la parte relacional, que debe trabajar en conjunto con Qdrant para gestionar lo que ella no puede. También aplique introducción de resúmenes e índices para los documentos lo que facilitaba la precisión de obtención de fragmentos de largo alcance.

Sobre los consejos, recomendaría siempre partir por la problemática a resolver, de otro modo la solución se adaptará al problema y no debiese ser así, sino al inverso, de otro modo no seria distinto a usar un producto general y creo que la ventaja de hacer la solución para un problema es que justamente este se hace para ese problema, al menos para un caso como el mío. 
